# NumPy — the Numeric Core

In notebook 01 we promised that every column of every `pandas` DataFrame is, underneath, a `numpy` array. This notebook makes that claim concrete.

NumPy gives you one data structure — the **`ndarray`** — and a small set of rules for working with it. Master four things and the rest of the PyData stack snaps into focus.

1. **What an `ndarray` actually is** — dtype plus shape plus a contiguous block of memory
2. **dtypes** — the typed-array contract
3. **Vectorization** — why you almost never write a `for` loop
4. **Broadcasting** — how arrays of different shapes still combine cleanly

That is the entire notebook.


## What is an `ndarray`?

An `ndarray` is three things bundled together.

- A **single contiguous block of memory** holding raw values (no Python objects, no boxing).
- A **dtype** — every value in that block is the same fixed-width type.
- A **shape** — a tuple like `(5,)` or `(3, 4)` that says how to interpret the block as rows and columns.

If you come from the JVM, this is the difference between an `ArrayList<Object>` and a `double[]`. Python's built-in `list` is the former — every element is a boxed object, the list stores pointers, and arithmetic has to chase pointers and unbox. NumPy's `ndarray` is the latter — one flat block of `double`s (or `int`s, or `bool`s), and arithmetic happens in native code, one whole array at a time.

The shape tuple is what lets the same memory block be read as 1D, 2D, or N-D. A 12-element block can be a length-12 vector, a 3×4 matrix, or a 2×2×3 cube — same bytes, different shape.


In [ ]:
import numpy as np

a = np.array([10, 20, 30, 40, 50])
print(a)
print("shape:  ", a.shape)
print("dtype:  ", a.dtype)
print("ndim:   ", a.ndim)
print("itemsize (bytes per element):", a.itemsize)


## Creating arrays — the common factories

You will only ever use a handful of constructors. Memorize these six.

| Call | What it gives you |
|---|---|
| `np.array([...])` | An array from a Python list or nested list |
| `np.zeros((m, n))` | All zeros, shape `(m, n)` |
| `np.ones((m, n))` | All ones, shape `(m, n)` |
| `np.full((m, n), v)` | All copies of `v`, shape `(m, n)` |
| `np.arange(start, stop, step)` | Evenly spaced integers (like Python's `range`) |
| `np.linspace(start, stop, n)` | `n` evenly spaced floats — *including both endpoints* |

`arange` is great for integer indices. `linspace` is what you want for plotting — give it the endpoints and a count, no fencepost errors.


In [ ]:
print("zeros:\n", np.zeros((2, 3)))
print("\nones:\n", np.ones((2, 3)))
print("\nfull:\n", np.full((2, 3), 7))
print("\narange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5))


## dtypes — the typed-array contract

Every `ndarray` has a single `dtype`. This is what lets NumPy stay fast: the engine knows it can do a hardware-level add on 64-bit floats without checking the type of each cell.

Common dtypes you will meet:

| dtype | What it holds |
|---|---|
| `int64` | 64-bit signed integer (the default on most systems) |
| `float64` | 64-bit IEEE float (the default for `np.zeros`, `np.ones`) |
| `bool` | True / False |
| `object` | Python object — escape hatch; avoid in numeric code |

If you assign a Python list with mixed types, NumPy picks the widest dtype that fits everything. Cast explicitly with `.astype(...)` when you want control.

**One gotcha:** integer dtypes overflow silently. An `int8` rolls over from 127 to -128 with no warning. Stick with `int64` unless you genuinely need to save memory.


In [ ]:
# Implicit dtype: NumPy picks the widest type that fits
print(np.array([1, 2, 3]).dtype)            # int64
print(np.array([1, 2, 3.0]).dtype)          # float64 — one float promoted the whole array
print(np.array([True, False, True]).dtype)  # bool

# Explicit cast
balances = np.array([45000, 82000, 120000], dtype=np.float64)
print(balances, balances.dtype)

# Overflow demo — int8 only holds -128 to 127
small = np.array([100, 100], dtype=np.int8)
print("int8 overflow:", small + small)  # wraps around silently


## Shape and reshape

A 12-element array can be read as `(12,)`, `(3, 4)`, `(4, 3)`, `(2, 6)`, or `(2, 2, 3)` — all the same underlying bytes, just a different view.

`reshape` does not move data. It changes how the array's strides interpret the block. That is why reshape is essentially free.

NumPy stores arrays in **C order** (row-major) by default: rows live contiguously in memory, and the last axis varies fastest as you walk through the block. This matches the JVM convention for flattening a `double[][]` and the SQL convention for laying out tabular data — so the mental model carries.


In [ ]:
a = np.arange(12)
print("flat:", a)

print("\nreshape(3, 4):")
print(a.reshape(3, 4))

print("\nreshape(4, 3):")
print(a.reshape(4, 3))

# Use -1 to let NumPy infer one dimension from the other
print("\nreshape(-1, 6):")
print(a.reshape(-1, 6))


## Vectorization — the headline

This is the single biggest reason NumPy exists.

**The Python way:** to multiply every element of a list by 2, you write a `for` loop. Each iteration goes through the interpreter — fetch the object, unbox the int, multiply, box the result, store the pointer. Slow.

**The NumPy way:** `arr * 2` on an `ndarray` calls a single compiled C loop that runs at memory-bandwidth speed. The interpreter is hit *once* per operation, not once per element.

The rule for writing fast NumPy code is short: **if you wrote a `for` loop over array elements, you wrote the wrong code.** Find the vectorized operation.


In [ ]:
n = 1_000_000
xs = list(range(n))
arr = np.arange(n)

# Python loop over a list
%timeit [x * 2 for x in xs]

# NumPy vectorized
%timeit arr * 2


## Broadcasting — how mismatched shapes line up

Vectorization handles arrays of the **same** shape. Broadcasting is the trick that makes arrays of **different** shapes work together — without copying the data.

The rule, in one sentence: NumPy compares shapes from the right; dimensions are compatible if they are equal, or if one of them is `1`. The `1`-dimension is conceptually stretched to match.

Three patterns cover 95% of broadcasting.

1. **Scalar + array** — the scalar is treated as if it were the same shape as the array.
2. **Row vector + matrix** — a `(n,)` (or `(1, n)`) row lines up with the last axis and is applied to every row.
3. **Column vector + matrix** — an `(m, 1)` column lines up with the first axis and is applied to every column.

The data is not actually duplicated in memory. NumPy fakes the stretch via strides, so broadcasting costs almost nothing.


In [ ]:
matrix = np.arange(12).reshape(3, 4)
print("matrix:\n", matrix)

# 1. Scalar broadcast
print("\nmatrix + 100:\n", matrix + 100)

# 2. Row vector broadcast — shape (4,) lines up with the last axis
row = np.array([1, 10, 100, 1000])
print("\nmatrix + row (added to every row):\n", matrix + row)

# 3. Column vector broadcast — shape (3, 1) lines up with the first axis
col = np.array([[1], [2], [3]])
print("\nmatrix + col (added to every column):\n", matrix + col)


## Aggregations along axes

`sum`, `mean`, `std`, `min`, `max` all take an `axis` argument that controls how the reduction sweeps the array.

- `axis=0` — collapse rows, keep columns. *"Sum down each column."*
- `axis=1` — collapse columns, keep rows. *"Sum across each row."*
- No axis — collapse everything to a single scalar.

The mnemonic: **the axis you name is the axis that goes away.**


In [ ]:
matrix = np.array([
    [10, 20, 30, 40],
    [50, 60, 70, 80],
    [90, 100, 110, 120],
])

print("total:               ", matrix.sum())
print("column sums (axis=0):", matrix.sum(axis=0))
print("row sums    (axis=1):", matrix.sum(axis=1))

print("\ncolumn means (axis=0):", matrix.mean(axis=0))
print("row stddevs  (axis=1):", matrix.std(axis=1).round(2))


## Fintech tie-in — applying interest rates

Picture five customers, each holding four products at Fintech Bank: home loan, car loan, personal loan, and gold loan. We have:

- A **5×4 matrix** of outstanding balances — one row per customer, one column per product.
- A **length-4 vector** of annual interest rates — one rate per product.

We want the annual interest each customer owes per product. That is a single broadcast multiplication — no loops, no joins, no copies. The `(5, 4)` matrix lines up with the `(4,)` rate vector on the last axis, and NumPy applies the right rate to the right column for every row.


In [ ]:
products = ["home_loan", "car_loan", "personal_loan", "gold_loan"]

# 5 customers, 4 products — outstanding balances in rupees
balances = np.array([
    [2_500_000,  450_000,  120_000,   80_000],
    [3_800_000,        0,  200_000,        0],
    [        0,  600_000,        0,  150_000],
    [1_900_000,  380_000,   50_000,        0],
    [        0,        0,  300_000,  220_000],
])

# annual rate per product
rates = np.array([0.085, 0.095, 0.140, 0.075])

annual_interest = balances * rates  # (5, 4) * (4,) → (5, 4)
print("annual interest per customer per product:")
print(annual_interest)

print("\ntotal annual interest per customer:", annual_interest.sum(axis=1))
print("total annual interest per product: ", annual_interest.sum(axis=0))


## What's next

NumPy gives you fast typed arrays. What it does not give you:

- **Labels.** You cannot ask "the row for customer 1042" — you only have integer positions.
- **Mixed types in one table.** A NumPy array is one dtype only; you cannot have name, age, and city in one structure cleanly.
- **Missing data.** No first-class `NA` marker that works across dtypes.

That is what **`pandas`** adds. Notebook 03 introduces `Series` and `DataFrame` — NumPy with labels, mixed types, and a proper missing-data story — and shows how to read real CSV, JSON, parquet, and SQL sources into them.

Two short from-memory exercises before you move on.

1. Build a `(4, 3)` matrix of card transaction amounts and a length-3 vector of category discount rates (e.g. `[0.02, 0.05, 0.10]`). Use broadcasting to get the discount each transaction qualifies for.
2. From the same matrix, compute the average transaction per category (`axis=0`) and per customer (`axis=1`).
